# level 5 L1000 annotation

In [1]:
from __future__ import annotations

from typing import Optional, Iterable
from pathlib import Path
import gzip
import shutil
import subprocess
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

In [20]:
import sys
sys.path.append("./op3_v2")

from src.utils.parsing_utils import *
from src.pseudobulking.common.pubchem import lookup_pubchem_cids
from src.pseudobulking.datasets.l1000.pubchem_imputation import pubchem_mapping_l1000
from src.pseudobulking.datasets.l1000.cellosaurus_annotation import annotate_cell_lines
from src.pseudobulking.datasets.l1000.donor_metadata_annotation import fetch_donor_info_from_cellosaurus
from src.pseudobulking.datasets.l1000.gene_annotation import fetch_ensg_ids
from src.pseudobulking.datasets.l1000.assembling import *

In [27]:
def _read_table(path: Path, **kwargs) -> pd.DataFrame:
    """
    Read a table file, handling gzipped files automatically.
    
    Attempts to read the file at the specified path. If not found, looks for
    a gzipped version (.gz) and reads it instead.
    
    Parameters
    ----------
    path : Path
        Path to the table file (CSV format)
    **kwargs
        Additional arguments passed to pd.read_csv()
        
    Returns
    -------
    pd.DataFrame
        Loaded table data
        
    Raises
    ------
    FileNotFoundError
        If neither the file nor its gzipped version exists
    """
    if not path.exists():
        gz = path.with_suffix(path.suffix + ".gz")
        if gz.exists():
            with gzip.open(gz, "rt") as fh:
                df = pd.read_csv(fh, **kwargs)
        else:
            raise FileNotFoundError(path)
    else:
        df = pd.read_csv(path, **kwargs)
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.lower()
    )
    return df

In [39]:
def download_l1000_files(data_root: Optional[str] = None, dataset: str = "l1000_phase1", skip_existing: bool = True) -> None:
    """
    Download L1000 Level 3 data files from GEO and CLUE.
    
    Downloads all required L1000 Level 3 files including:
    - GCTX expression file (48.8 GB)
    - Instance metadata
    - Cell line metadata
    - Perturbagen metadata
    - Gene annotations
    
    Parameters
    ----------
    data_root : str, optional
        Root directory for data files. If None, uses default from define_paths()
    dataset : str, default="l1000_phase1"
        Dataset name: "l1000_phase1" or "l1000_phase2"
    skip_existing : bool, default=True
        If True, skips downloading files that already exist
        
    Raises
    ------
    subprocess.CalledProcessError
        If download command fails
    """
    if data_root is None:
        paths = define_paths(dataset=dataset)
        data_root = Path(paths["level5_gctx"]).parent
    else:
        data_root = Path(data_root)
    
    data_root.mkdir(parents=True, exist_ok=True)
    
    manifest_df = get_download_manifest(data_root, dataset=dataset)
    
    logger.info(f"Downloading L1000 Level 3 files to: {data_root}")
    
    for _, row in manifest_df.iterrows():
        file_path = row["path"]
        
        if skip_existing and file_path.exists():
            logger.info(f"  {row['file']} already exists, skipping")
            continue
        
        logger.info(f"  Downloading {row['file']} ({row['size']})...")
        cmd = f"curl -L '{row['url']}' -o {file_path}"
        
        try:
            subprocess.run(cmd, shell=True, check=True)
            logger.info(f"    Downloaded {row['file']}")
        except subprocess.CalledProcessError as e:
            logger.error(f"    Failed to download {row['file']}: {e}")
            raise
    
    logger.info("All downloads complete")

def decompress_l1000_files(data_root: Optional[str] = None, dataset: str = "l1000_phase1") -> None:
    """
    Decompress gzipped L1000 data files.
    
    Decompresses all .gz files in the data directory, including the large GCTX
    expression file. The GCTX decompression may take several minutes.
    
    Parameters
    ----------
    data_root : str, optional
        Root directory containing compressed files. If None, uses default from define_paths()
    dataset : str, default="l1000_phase1"
        Dataset name: "l1000_phase1" or "l1000_phase2"
        
    Raises
    ------
    FileNotFoundError
        If compressed files are not found
    """
    if data_root is None:
        paths = define_paths(dataset=dataset)
        data_root = Path(paths["level5_gctx"]).parent
    else:
        data_root = Path(data_root)
    
    paths = define_paths(str(data_root), dataset=dataset)
    to_decompress = []
    already_done = []
    
    for key, path in paths.items():
        if key.endswith("_gz"):
            continue
        
        path = Path(path)
        if not path.suffix == ".gz":
            gz_candidate = path.with_suffix(path.suffix + ".gz")
            if gz_candidate.exists() and not path.exists():
                to_decompress.append((key, gz_candidate, path))
            elif path.exists():
                already_done.append((key, path))
    
    if already_done:
        logger.info(f"{len(already_done)} file(s) already decompressed")
        for key, path in already_done:
            logger.info(f"  - {key}: {path.name}")
    
    if to_decompress:
        logger.info(f"Decompressing {len(to_decompress)} file(s)...")
        for key, gz_path, target_path in to_decompress:
            logger.info(f"  - {key}: {gz_path.name} -> {target_path.name}")
            if "gctx" in key.lower():
                logger.info("    (GCTX is large, this may take a few minutes...)")
            
            with gzip.open(gz_path, "rb") as src, open(target_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
            logger.info(f"    Done")
        
        logger.info(f"All {len(to_decompress)} file(s) successfully decompressed")
    else:
        if not already_done:
            logger.warning("No compressed files found. Download them first using download_l1000_files()")
        else:
            logger.info("All files are already decompressed")


def get_download_manifest(data_root: Path, dataset: str = "l1000_phase1") -> pd.DataFrame:
    """
    Get manifest of L1000 Level 3 files to download.
    
    Returns a DataFrame with download information for all required L1000 Level 3
    files from GEO and CLUE resources.
    
    Parameters
    ----------
    data_root : Path
        Root directory where files will be downloaded
        
    Returns
    -------
    pd.DataFrame
        Download manifest with columns: file, kind, size, url, path, notes, curl_example
    """
    if dataset == "l1000_phase1":
        download_manifest = [
            {
                "file": "GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx.gz",
                "kind": "expression",
                "size": "...",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx.gz",
                "notes": "Processed expression"
            },
            {
                "file": "GSE92742_Broad_LINCS_sig_info.txt.gz", 
                "kind": "metadata", 
                "size": "~10.6 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_sig_info.txt.gz",
                "notes": "Signatures information"
            },
            {
                "file": "GSE92742_Broad_LINCS_inst_info.txt.gz",
                "kind": "metadata",
                "size": "~150 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_inst_info.txt.gz",
                "notes": "Instance-level annotations"
            },
            {
                "file": "GSE92742_Broad_LINCS_cell_info.txt.gz",
                "kind": "metadata",
                "size": "<10 KB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_cell_info.txt.gz",
                "notes": "Cell line annotations"
            },
            {
                "file": "GSE92742_Broad_LINCS_pert_info.txt.gz",
                "kind": "metadata",
                "size": "~5 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_pert_info.txt.gz",
                "notes": "Perturbagen annotations"
            },
            {
                "file": "GSE92742_Broad_LINCS_gene_info.txt.gz",
                "kind": "metadata",
                "size": "<1 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_gene_info.txt.gz",
                "notes": "Gene annotations"
            }
        ]
    elif dataset == "l1000_phase2":
        download_manifest = [
                {
                "file": "GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx.gz",
                "kind": "expression",
                "size": "12.6 GB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx.gz",
                "notes": "Raw epsilon (landmark genes)"
            },
               {
                "file": "GSE70138_Broad_LINCS_sig_info_2017-03-06.txt.gz", 
                "kind": "metadata", 
                "size": "~1.9 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_sig_info_2017-03-06.txt.gz",
                "notes": "Signatures information"
            },
            {
                "file": "GSE70138_Broad_LINCS_inst_info_2017-03-06.txt.gz",
                "kind": "metadata",
                "size": "~150 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_inst_info_2017-03-06.txt.gz",
                "notes": "Instance-level annotations"
            },
            {
                "file": "GSE70138_Broad_LINCS_cell_info_2017-04-28.txt.gz",
                "kind": "metadata",
                "size": "<10 KB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_cell_info_2017-04-28.txt.gz",
                "notes": "Cell line annotations"
            },
            {
                "file": "GSE70138_Broad_LINCS_pert_info.txt.gz",
                "kind": "metadata",
                "size": "~5 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_pert_info.txt.gz",
                "notes": "Perturbagen metadata"
            },
            {
                "file": "GSE70138_Broad_LINCS_gene_info_2017-03-06.txt.gz",
                "kind": "metadata",
                "size": "~210 KB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_gene_info_2017-03-06.txt.gz",
                "notes": "Landmark gene annotations"
            }
            
        ]
    else:
        raise ValueError(f"Invalid dataset: {dataset}")
    
    manifest_df = pd.DataFrame(download_manifest)
    manifest_df["path"] = manifest_df["file"].apply(lambda f: data_root / f)
    manifest_df["curl_example"] = manifest_df.apply(
        lambda row: f"curl -L '{row['url']}' -o {row['path']}",
        axis=1
    )
    
    return manifest_df

def check_l1000_files(data_root: Optional[str] = None, dataset: str = "l1000_phase1") -> dict:
    """
    Check status of L1000 data files.
    
    Checks which files are missing, compressed, or ready to use.
    
    Parameters
    ----------
    data_root : str, optional
        Root directory to check. If None, uses default from define_paths()
    dataset : str, default="l1000_phase1"
        Dataset name: "l1000_phase1" or "l1000_phase2"
        
    Returns
    -------
    dict
        Dictionary with keys 'missing', 'compressed', 'ready' containing lists of
        (key, path) tuples for each category
    """
    if data_root is None:
        paths = define_paths(dataset=dataset)
        data_root = Path(paths["sig"]).parent
    else:
        data_root = Path(data_root)
    
    paths = define_paths(str(data_root), dataset=dataset)
    
    missing = []
    compressed = []
    ready = []
    
    for key, path in paths.items():
        if key.endswith("_gz"):
            continue
        
        # Skip pubchem cache JSON files
        path_str = str(path)
        if "pubchem_cache" in key.lower() and path_str.endswith(".json"):
            continue

        # Skip hgnc cache JSON files
        if "hgnc_cache" in key.lower() and path_str.endswith(".json"):
            continue
        
        path = Path(path)
        if path.suffix == ".gz":
            if not path.exists():
                missing.append((key, path))
        else:
            gz_candidate = path.with_suffix(path.suffix + ".gz")
            if path.exists():
                ready.append((key, path))
            elif gz_candidate.exists():
                compressed.append((key, gz_candidate))
            else:
                missing.append((key, path))
    
    # Log status
    if missing:
        logger.warning(f"{len(missing)} file(s) missing:")
        for key, path in missing:
            logger.warning(f"  - {key}: {path.name}")
        logger.info("  Run download_l1000_files() to download them")
    else:
        logger.info("All required files are present")
    
    if compressed:
        logger.warning(f"{len(compressed)} file(s) still compressed:")
        for key, path in compressed:
            logger.warning(f"  - {key}: {path.name}")
        logger.info("  Run decompress_l1000_files() to extract them")
    else:
        if not missing:
            logger.info("All files are uncompressed and ready to use")
    
    return {
        "missing": missing,
        "compressed": compressed,
        "ready": ready
    }


def standardize_sig(sig: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize signature metadata.
    
    This function standardizes the signature ID column to 'lincs_sig_id' and adds plate
    information if missing by parsing the signature ID.
    
    Parameters
    ----------
    sig : pd.DataFrame
        Instance metadata with sig_id, sample_id, or distil_id column
        
    Returns
    -------
    pd.DataFrame
        Standardized dataframe with 'lincs_sig_id' as index and column, and 'det_plate' column added if missing
    """
    df = sig.copy()
    
    # Find ID column
    for id_col in ["sig_id", "sample_id", "distil_id"]:
        if id_col in df.columns:
            df = df.rename(columns={id_col: "lincs_sig_id"})
            break
    else:
        raise KeyError("sig info missing sig_id/sample_id/distil_id")
    
    # Add plate info if missing
    #if 'det_plate' not in df.columns:
    #    df['det_plate'] = df['lincs_sig_id'].str.split(':').str[0]
    
    return df.set_index("lincs_sig_id", drop=False)








def build_obs_dataframe(sig: pd.DataFrame, dataset: str = "l1000_phase1") -> pd.DataFrame:
    """
    Build standardized .obs dataframe from signature metadata.
    Sticked to https://lamin.ai/laminlabs/pertdata/transform/REAvqqdo3sbH0000
    
    Parameters
    ----------
    sig : pd.DataFrame
        Instance metadata dataframe with LINCS information
        
    Returns
    -------
    pd.DataFrame
        Standardized obs dataframe with pseudobulk schema
    """
    obs = pd.DataFrame(index=sig.index)
    
    # Copy identifier columns
    for extra_id in ("lincs_sig_id", "distil_id", "inst_id", "sample_id", "sig_id"):
        if extra_id in sig.columns:
            obs[extra_id] = sig[extra_id].astype("string")
    
    # Map perturbation types to standard schema
    PERT_TYPE_MAP = {
        "trt_cp": "compound",
        "trt_lig": "biologic",
        "trt_sh": "genetic",
        "trt_oe": "genetic",
        "trt_oe.mut": "genetic",
        "trt_xpr": "genetic",
        "ctl_vehicle": "compound",
        "trt_poscon": "compound",
        "ctl_vector": "genetic",
        "ctl_untrt": "biologic"
        
    }

    
    # Build standard obs columns
    obs["plate"] = sig.get("det_plate", None)
    obs["well"] = sig.get("rna_well", sig.get("det_well", None))
    obs["cell_type"] = sig.get("cellinfo_cell_id_mixed", sig.get("cell_id", None)).fillna(sig.get("cell_id", None))
    obs["perturbagen"] = sig.get("pert_iname", None)
    obs["pert_type"] = sig["pert_type"].map(PERT_TYPE_MAP)
    obs["is_control"] = sig["pert_type"].str.startswith("ctl")
    obs["pert_dose_uM"] = sig["pert_dose_um"].astype(float)
    obs.loc[(obs['perturbagen']=='DMSO') & (obs['is_control']==True), 'pert_dose_uM'] = 0
    obs['pert_dose'] = sig['pert_dose'].astype(str) + ' ' + sig['pert_dose_unit'].astype(str)
    obs["pert_time_h"] = sig["pert_time_h"].astype(float)
    obs["suspension_type"] = "cell"
    obs["tissue"] = sig.get("cellinfo_primary_site", "unknown")
    obs["tissue_type"] = "cell culture"
    obs["disease"] = sig.get("cellinfo_subtype", "unknown")
    obs["library"] = None
    obs["stimulation"] = None
    obs["guide"] = None

    if dataset == "l1000_phase1":
        obs["dataset"] = "LINCS_phase1_level3_epsilon"
    elif dataset == "l1000_phase2":
        obs["dataset"] = "LINCS_phase2_level3"
    else:
        raise ValueError(f"Invalid dataset: {dataset}")

    obs["assay"] = "L1000 mRNA profiling assay"
    obs["development_stage"] = sig.apply(build_development_stage, axis=1)
    obs["organism"] = "human"
    if 'cellinfo_donor_sex' in sig.columns:
        obs["sex"] = sig["cellinfo_donor_sex"].map({"M": "male", "F": "female"})
    else:
        obs["sex"] = "unknown"
        
    obs["self_reported_ethnicity"] = sig.get("cellinfo_donor_ethnicity", "unknown")
    if 'pubchem_cid' in sig.columns:
        sig['pubchem_cid'] = pd.to_numeric(sig['pubchem_cid'], errors='coerce').fillna(-666).astype('int64')
    obs["pubchem_cid"] = sig.get("pubchem_cid", None)
    obs["psbulk_cells"] = None
    obs["psbulk_counts"] = None
    
    # Add metadata columns
    obs["lincs_sig_id"] = sig.index.astype("string")
    obs["source_gctx"] = sig["source_gctx"].astype("string")
    
    # Create composite sample_id
    obs["sample_id"] = (
        obs["plate"].astype(str).str.replace(" ", "", regex=False) + "_" +
        obs["well"].astype(str).str.replace(" ", "", regex=False) + "_" +
        obs["perturbagen"].astype(str).str.replace(" ", "_", regex=False) + "_" +
        obs["cell_type"].astype(str).str.replace(" ", "_", regex=False)
    )

    obs["pert_type_init"] = sig["pert_type_pert"].copy()
    # Clean up missing values and duplicates
    obs = obs.replace({-666: None, '-666': None, 'None': None, 'nan': None, '<NA>': None})
    #obs = obs[~obs["sample_id"].duplicated(keep="first")]
    obs = obs.set_index("sig_id", drop=True)
    obs = materialize_string_columns(obs)
    return obs



def define_obs_schema() -> list:
    """
    Define the strict obs schema for L1000 pseudobulk data.
    
    Returns
    -------
    list
        List of tuples defining the schema: (column_name, dtype, description)
    """
    return [
        ("sample_id", "category", "ID of the observation: plate + well + cell_type + perturbagen"),
        ("plate", "category", "Assay plate identifier (det_plate)"),
        ("well", "category", "Well ID on the RNA plate (rna_well)"),
        ("cell_type", "category", "Cell line / cell_id"),
        ("perturbagen", "category", "Human-readable perturbagen label"),
        ("pert_type", "category", "Perturbation class"),
        ("is_control", "category", "True/False for controls"),
        ("pert_dose_uM", "float64", "Dose in micromolar"),
        ("pert_time_h", "float64", "Exposure time in hours"),
        ("suspension_type", "category", "Growth pattern"),
        ("tissue", "category", "Primary tissue/site"),
        ("tissue_type", "category", "Sample type"),
        ("disease", "category", "Disease/subtype"),
        ("library", "category", "Library/release"),
        ("stimulation", "category", "High-level stimulus"),
        ("guide", "category", "A guide RNA directs the CRISPR system"),
        ("dataset", "category", "Dataset label"),
        ("assay", "category", "Assay label"),
        ("development_stage", "category", "Derived from donor age"),
        ("organism", "category", "Organism"),
        ("sex", "category", "Donor sex"),
        ("self_reported_ethnicity", "category", "Donor ethnicity"),
        ("pubchem_cid", "category", "PubChem CID"),
        ("psbulk_cells", "int64", "Total #cells contributing (if no info - then -666)"),
        ("psbulk_counts", "int64", "Total #counts contributing (if no info - then -666)"),
        ("distil_id", "category", "distil_id"),
        ("sig_id", "category", "signature id"),
        ("pert_type_init", "category", "initial perturbation type"),
        ("pert_dose", "category", "initial perturbation type")
    ]


def define_paths(data_root: Optional[str] = None, dataset: str = "l1000_phase1") -> dict:
    """
    Define file paths for L1000 Level 3 data.
    
    Parameters
    ----------
    data_root : str, optional
        Root directory containing L1000 data files.
        Defaults to './lincs_data'
        
    Returns
    -------
    dict
        Dictionary mapping file identifiers to Path objects
    """
    if data_root is None:
        data_root = './lincs_data'
    
    data_root = Path(data_root)
    processed_dir = data_root / "processed"
    processed_dir.mkdir(exist_ok=True)
    
    if dataset == "l1000_phase1":
        return {
                "level5_gctx": data_root / "GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx",
                "level5_gctx_gz": data_root / "GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx.gz",
                "instinfo": data_root / "GSE92742_Broad_LINCS_inst_info.txt",
                "siginfo": data_root / "GSE92742_Broad_LINCS_sig_info.txt",
                "cellinfo": data_root / "GSE92742_Broad_LINCS_cell_info.txt",
                "pert_info": data_root / "GSE92742_Broad_LINCS_pert_info.txt",
                "geneinfo": data_root / "GSE92742_Broad_LINCS_gene_info.txt",
                "pubchem_cache": processed_dir / "pubchem_cache.json",
                "hgnc_cache": processed_dir / "hgnc_cache.json"
                
        }
    elif dataset == "l1000_phase2":
        return {
                "level5_gctx": data_root / "GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx",
                "level5_gctx_gz": data_root / "GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx.gz",
                "instinfo": data_root / "GSE70138_Broad_LINCS_inst_info_2017-03-06.txt",
                "siginfo": data_root / "GSE70138_Broad_LINCS_sig_info_2017-03-06.txt",
                "cellinfo": data_root / "GSE70138_Broad_LINCS_cell_info_2017-04-28.txt",
                "pert_info": data_root / "GSE70138_Broad_LINCS_pert_info.txt",
                "geneinfo": data_root / "GSE70138_Broad_LINCS_gene_info_2017-03-06.txt",
                "pubchem_cache": processed_dir / "pubchem_cache.json",
                "hgnc_cache": processed_dir / "hgnc_cache.json",
        }
    else:
        raise ValueError(f"Invalid dataset: {dataset}")

def add_alternative_identifiers(sig_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize signature metadata and add alternative identifier columns.
    
    This function standardizes the signature ID column and preserves alternative
    identifiers (distil_id, sig_id, sample_id, sig_id) for cross-referencing.
    
    Parameters
    ----------
    sig_raw : pd.DataFrame
        Raw signature/sample metadata
        
    Returns
    -------
    pd.DataFrame
        Instance metadata with standardized lincs_sig_id index
        and additional identifier columns
    """
    logger.info('  Processing signature metadata')
    sig = standardize_sig(sig_raw)
    
    # Keep alternative identifier columns
    sig_raw_tmp = sig_raw.copy()
    if 'lincs_sig_id' not in sig_raw_tmp.columns:
        if 'sample_id' in sig_raw_tmp.columns:
            sig_raw_tmp['lincs_sig_id'] = sig_raw_tmp['sample_id']
        elif 'sig_id' in sig_raw_tmp.columns:
            sig_raw_tmp['lincs_sig_id'] = sig_raw_tmp['sig_id']
        else:
            raise ValueError("signature metadata must have lincs_sig_id, sample_id, or sig_id column")
    
    sig_raw_indexed = sig_raw_tmp.set_index("lincs_sig_id", drop=False)
    
    for extra_id in ("distil_id", "sig_id", "sample_id", "sig_id"):
        if extra_id in sig_raw_indexed.columns:
            sig[extra_id] = sig_raw_indexed.loc[sig.index, extra_id].astype("string")
    
    return sig





def enrich_sig_metadata(sig: pd.DataFrame, 
                            cellinfo: pd.DataFrame,
                            pert_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Enrich instance metadata by merging with cell and perturbation info.
    
    Parameters
    ----------
    sig : pd.DataFrame
        Processed signature metadata
    cellinfo : pd.DataFrame
        Cell line information
    pert_raw : pd.DataFrame
        Perturbation information
        
    Returns
    -------
    pd.DataFrame
        Enriched signature metadata with merged information
    """
    def fix_cell_line_annotation(df: pd.DataFrame) -> pd.DataFrame:
        mask = df['cell_id'] == 'SNUC4'
        df.loc[mask, 'cellinfo_subtype'] = 'colon adenocarcinoma'
        df.loc[mask, 'cellinfo_donor_age'] = '35'
        df.loc[mask, 'cellinfo_donor_sex'] = 'M'
        df.loc[mask, 'cellinfo_donor_ethnicity'] = 'Korean'
        df.loc[mask, 'cellinfo_cellosaurus_id'] = 'CVCL_5111'
        df.loc[mask, 'cellinfo_cell_id_mixed'] = 'CVCL_5111'
        return df
    
    pert_cols = ["pert_id", "pert_iname", "pert_type", "pubchem_cid"]
    sig = sig.merge(cellinfo, how="left", left_on="cell_id", right_index=True)    
    sig = sig.merge(pert_raw[pert_cols].drop_duplicates("pert_id"), 
                     how="left", on="pert_id", suffixes=("", "_pert"))

    sig = fix_cell_line_annotation(sig).copy()
    
    return sig


def process_sig_metadata(sig_raw: pd.DataFrame,
                            cellinfo: pd.DataFrame,
                            pert_raw: pd.DataFrame,
                            config: dict,
                            paths: dict,
                            max_samples: int = 1000) -> pd.DataFrame:
    """
    Process signature metadata through complete pipeline.
    
    This function handles the full signature metadata processing pipeline:
    1. Add alternative identifiers (distil_id, sig_id, sample_id, sig_id)
    2. Enrich with cell and perturbation information
    3. Standardize dose units to micromolar
    4. Standardize time units to hours
    5. Apply filters (perturbation types, controls, subsampling)
    6. Add source file information
    
    Parameters
    ----------
    sig_raw : pd.DataFrame
        Raw signature/sample metadata
    cellinfo : pd.DataFrame
        Cell line information
    pert_raw : pd.DataFrame
        Perturbation information
    config : dict
        Configuration with filter settings
    paths : dict
        Dictionary of file paths
        
    Returns
    -------
    pd.DataFrame
        Fully processed and filtered signature metadata
    """
    sig = add_alternative_identifiers(sig_raw)
    sig = enrich_sig_metadata(sig, cellinfo, pert_raw)
    sig = standardize_dose(sig)
    sig = standardize_time(sig)
    
    # Apply filters
    if config["perturbation_types_to_keep"] is not None:
        sig = sig[sig["pert_type"].isin(config["perturbation_types_to_keep"])]

    if config["control"] is not None:
        sig = sig[(sig['pert_type'].str.startswith('ctl') & sig['pert_iname'].isin(config['control'])) |
                    (~sig['pert_type'].str.startswith('ctl'))]

    cell_ids_with_controls = set(sig[sig['pert_type'].str.startswith('ctl')]['cell_id'].unique())
    cell_ids_with_compounds = set(sig[~sig['pert_type'].str.startswith('ctl')]['cell_id'].unique())
    valid_cell_ids = cell_ids_with_controls & cell_ids_with_compounds
    if len(valid_cell_ids) == 0:
        raise ValueError("No cell lines found with both controls and compounds")

    sig = sig[sig['cell_id'].isin(valid_cell_ids)].copy()

    if config["subsampling"]:
        sig = sig.sample(max_samples, random_state=0)
    
    # Add source file information
    sig["source_gctx"] = str(paths["level5_gctx"])
    
    return sig


def load_metadata_tables(paths: dict) -> tuple:
    """
    Load L1000 metadata tables from files.
    
    Parameters
    ----------
    paths : dict
        Dictionary of file paths from define_paths()
        
    Returns
    -------
    tuple
        (sig_raw, cellinfo_raw, pert_raw, geneinfo)
        - sig_raw: Instance/sample information
        - cellinfo_raw: Cell line information
        - pert_raw: Perturbation information
        - geneinfo: Gene annotations (Level 3)
    """
    logger.info('  Loading metadata tables')
    inst_raw = _read_table(paths["instinfo"], sep="\t", low_memory=False)
    sig_raw = _read_table(paths["siginfo"], sep="\t", low_memory=False)
    cellinfo_raw = _read_table(paths["cellinfo"], sep="\t")
    pert_raw = _read_table(paths["pert_info"], sep="\t")
    geneinfo = _read_table(paths["geneinfo"], sep="\t")
    
    
    # Log loaded table sizes
    logger.info(f"    Loaded {len(sig_raw):,} signatures")
    logger.info(f"    Loaded {len(cellinfo_raw):,} cell lines")
    logger.info(f"    Loaded {len(pert_raw):,} perturbations")
    logger.info(f"    Loaded {len(geneinfo):,} genes")
    
    return inst_raw, sig_raw, cellinfo_raw, pert_raw, geneinfo





In [29]:
config = {
        'data_root': './lincs_data',
        'output_file': 'level5_phase1_not_filtered.h5ad',
        'perturbation_types_to_keep': None,
        'control': None,
        'full_gene_matrix': False,
        'subsampling': False,
        'annotate_pubchem': True,
        'download_if_missing': True,
        'dataset': 'l1000_phase1',
    }

In [30]:
data_root = './lincs_data'
padata = ad.AnnData()

In [31]:
logger.info('Applying L1000-specific processing')

if not HAS_CMAPPY:
    raise ImportError("cmapPy is required for L1000 processing. Install via: pip install cmapPy")

# Build configuration
CONFIG = build_config(config)
download_if_missing = CONFIG.get("download_if_missing", True)

# Define file paths
PATHS = define_paths(data_root, dataset=CONFIG.get("dataset"))



2026-01-29 19:54:26 | [INFO] Applying L1000-specific processing


In [32]:
# Check data availability and download if needed
if download_if_missing:
        logger.info("Checking data availability...")
        status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        
        if status['missing']:
            logger.info(f"Downloading {len(status['missing'])} missing file(s)...")
            download_l1000_files(data_root, dataset=CONFIG.get("dataset"), skip_existing=True)
            status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        
        
        if status['compressed']:
            logger.info(f"Decompressing {len(status['compressed'])} compressed file(s)...")
            decompress_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        
        # Verify all files are ready
        final_status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        if final_status['missing'] or final_status['compressed']:
            raise FileNotFoundError(
                f"Required L1000 data files are still missing or compressed after download attempt. "
                f"Missing: {len(final_status['missing'])}, Compressed: {len(final_status['compressed'])}"
            )
        logger.info("All required data files are available")

FULL_GENE_MATRIX = CONFIG["full_gene_matrix"]

2026-01-29 19:54:26 | [INFO] Checking data availability...
2026-01-29 19:54:26 | [INFO] All required files are present
2026-01-29 19:54:26 | [INFO] All files are uncompressed and ready to use
2026-01-29 19:54:26 | [INFO] All required files are present
2026-01-29 19:54:26 | [INFO] All files are uncompressed and ready to use
2026-01-29 19:54:26 | [INFO] All required data files are available


In [33]:
# Load metadata tables
inst_raw, sig_raw, cellinfo_raw, pert_raw, geneinfo = load_metadata_tables(PATHS)

2026-01-29 19:54:27 | [INFO]   Loading metadata tables
2026-01-29 19:54:40 | [INFO]     Loaded 473,647 signatures
2026-01-29 19:54:40 | [INFO]     Loaded 98 cell lines
2026-01-29 19:54:40 | [INFO]     Loaded 51,383 perturbations
2026-01-29 19:54:40 | [INFO]     Loaded 12,328 genes


In [34]:
NAME_TO_CVCL = {
        "HA1E": "CVCL_VU89",
        "HEK293T": "CVCL_0063",
        "HS27A": "CVCL_3719",
        "FIBRNPC": "CVCL_UK07",
        "U266": "CVCL_0566",
        "HUES3": "CVCL_B161",
        "HUVEC": "CVCL_2959",
        "H1299": "CVCL_0060",
        "HL60": "CVCL_0002",
        "SKBR3": "CVCL_0033",
        "ASC": "CVCL_U602"
    }

In [35]:
cellinfo = process_cellinfo(cellinfo_raw)

2026-01-29 19:55:07 | [INFO] Annotating cell lines with Cellosaurus IDs
2026-01-29 19:55:08 | [INFO] Annotating 81 unique cell types
2026-01-29 19:55:14 | [INFO] Processed 10/81 cell types (10 mapped so far)
2026-01-29 19:55:20 | [INFO] Processed 20/81 cell types (20 mapped so far)
2026-01-29 19:55:25 | [INFO] Processed 30/81 cell types (30 mapped so far)
2026-01-29 19:55:30 | [INFO] Processed 40/81 cell types (40 mapped so far)
2026-01-29 19:55:35 | [INFO] Processed 50/81 cell types (49 mapped so far)
2026-01-29 19:55:39 | [INFO] Processed 60/81 cell types (59 mapped so far)
2026-01-29 19:55:44 | [INFO] Processed 70/81 cell types (69 mapped so far)
2026-01-29 19:55:50 | [INFO] Processed 80/81 cell types (74 mapped so far)
2026-01-29 19:55:50 | [INFO] Completed: 74/81 cell types successfully mapped to Cellosaurus IDs
2026-01-29 19:55:51 | [INFO] Fetching donor information from Cellosaurus API
2026-01-29 19:55:51 | [INFO] Fetching donor information for 74 unique Cellosaurus IDs
2026-01-

In [36]:
# Process perturbation metadata
pert_raw = process_pert_metadata(pert_raw, sig_raw)

In [37]:
# Annotate compounds with PubChem CIDs (optional)
if CONFIG.get("annotate_pubchem", False):
    logger.info("Mapping compounds to PubChem CIDs")
    pert_raw = annotate_pubchem_cids(pert_raw, PATHS, config=CONFIG)

2026-01-29 19:57:15 | [INFO] Mapping compounds to PubChem CIDs


2026-01-29 19:57:15 | [WARNING] Error standardizing pubchem_cid: invalid literal for int() with base 10: 'MLS003116075'


2026-01-29 19:57:16 | [INFO] Loaded cache from lincs_data/processed/pubchem_cache.json with 21402 entries
2026-01-29 19:57:16 | [INFO] Processing 20416 compounds
2026-01-29 19:57:16 | [INFO] Processed 50/20416 compounds (50 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 100/20416 compounds (100 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 150/20416 compounds (150 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 200/20416 compounds (200 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 250/20416 compounds (250 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 300/20416 compounds (300 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 350/20416 compounds (350 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 400/20416 compounds (400 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 450/20416 compounds (450 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 500/20416 compounds (500 mapped so far)
2026-01-29 19:57:16 | [INFO] Processed 550/20416 c

[19:57:17] SMILES Parse Error: syntax error while parsing: -666
[19:57:17] SMILES Parse Error: check for mistakes around position 1:
[19:57:17] -666
[19:57:17] ^
[19:57:23] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'


2026-01-29 19:57:25 | [INFO] Processed 1400/20416 compounds (1399 mapped so far)
2026-01-29 19:57:25 | [INFO] Processed 1450/20416 compounds (1449 mapped so far)
2026-01-29 19:57:25 | [INFO] Processed 1500/20416 compounds (1499 mapped so far)
2026-01-29 19:57:25 | [INFO] Processed 1550/20416 compounds (1549 mapped so far)
2026-01-29 19:57:25 | [INFO] Processed 1600/20416 compounds (1599 mapped so far)
2026-01-29 19:57:25 | [INFO] Processed 1650/20416 compounds (1649 mapped so far)
2026-01-29 19:57:25 | [INFO] Processed 1700/20416 compounds (1699 mapped so far)
2026-01-29 19:57:25 | [INFO] Processed 1750/20416 compounds (1749 mapped so far)


...

2026-01-29 19:57:41 | [INFO] Processed 20150/20416 compounds (20128 mapped so far)
2026-01-29 19:57:41 | [INFO] Processed 20200/20416 compounds (20178 mapped so far)
2026-01-29 19:57:41 | [INFO] Processed 20250/20416 compounds (20228 mapped so far)
2026-01-29 19:57:41 | [INFO] Processed 20300/20416 compounds (20278 mapped so far)
2026-01-29 19:57:41 | [INFO] Processed 20350/20416 compounds (20328 mapped so far)


[19:57:41] SMILES Parse Error: syntax error while parsing: -666
[19:57:41] SMILES Parse Error: check for mistakes around position 1:
[19:57:41] -666
[19:57:41] ^
[19:57:41] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[19:57:42] SMILES Parse Error: syntax error while parsing: -666
[19:57:42] SMILES Parse Error: check for mistakes around position 1:
[19:57:42] -666
[19:57:42] ^
[19:57:42] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[19:57:42] SMILES Parse Error: syntax error while parsing: -666
[19:57:42] SMILES Parse Error: check for mistakes around position 1:
[19:57:42] -666
[19:57:42] ^
[19:57:42] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[19:57:43] SMILES Parse Error: syntax error while parsing: restricted
[19:57:43] SMILES Parse Error: check for mistakes around position 1:
[19:57:43] restricted
[19:57:43] ^
[19:57:43] SMILES Parse Error: Failed parsing SMILES 'restricted' for input: 'restricted'
[19:57:43] SMILE

2026-01-29 19:57:54 | [INFO] Processed 20400/20416 compounds (20356 mapped so far)


[19:57:54] SMILES Parse Error: syntax error while parsing: -666
[19:57:54] SMILES Parse Error: check for mistakes around position 1:
[19:57:54] -666
[19:57:54] ^
[19:57:54] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[19:57:54] SMILES Parse Error: syntax error while parsing: -666
[19:57:54] SMILES Parse Error: check for mistakes around position 1:
[19:57:54] -666
[19:57:54] ^
[19:57:54] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[19:57:55] SMILES Parse Error: syntax error while parsing: -666
[19:57:55] SMILES Parse Error: check for mistakes around position 1:
[19:57:55] -666
[19:57:55] ^
[19:57:55] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[19:57:55] SMILES Parse Error: syntax error while parsing: -666
[19:57:55] SMILES Parse Error: check for mistakes around position 1:
[19:57:55] -666
[19:57:55] ^
[19:57:55] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[19:57:56] SMILES Parse Error: syntax er

2026-01-29 19:57:58 | [INFO] Mapped 20364 out of 20416 compounds to PubChem CIDs


In [41]:
# Process signature metadata
sig = process_sig_metadata(sig_raw, cellinfo, pert_raw, CONFIG, PATHS)

2026-01-29 19:59:01 | [INFO]   Processing signature metadata


In [42]:
logger.info('  Building .obs dataframe')
obs = build_obs_dataframe(sig, dataset=CONFIG.get("dataset"))

2026-01-29 19:59:46 | [INFO]   Building .obs dataframe


In [43]:
obs

,lincs_sig_id,distil_id,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_dose,...,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,source_gctx,sample_id,pert_type_init
sig_id,,,,,,,,,,,,,,,,,,,,,
AML001_CD34_24H:A05,0,AML001_CD34_24H_X1_F1B10:A05,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:A06,1,AML001_CD34_24H_X3_F1B10:A06,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:B05,2,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:B06,3,AML001_CD34_24H_X3_F1B10:B06,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:BRD-A03772856:0.37037,4,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...,None,None,CD34,BRD-A03772856,compound,False,0.37037,0.37037 µM,...,unknown,human,NaN,None,3237298,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_BRD-A03772856_CD34,trt_cp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TAK004_U2OS_96H:TRCN0000370007:1,473642,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:J10|TAK004_U...,None,None,CVCL_0042,WWTR1,genetic,False,NaN,1.0 µL,...,15-year-old stage,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_WWTR1_CVCL_0042,trt_sh
TAK004_U2OS_96H:TRCN0000370678:1,473643,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:E02|TAK004_U...,None,None,CVCL_0042,GRB10,genetic,False,NaN,1.0 µL,...,15-year-old stage,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_GRB10_CVCL_0042,trt_sh
TAK004_U2OS_96H:TRCN0000370697:1,473644,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:A18|TAK004_U...,None,None,CVCL_0042,GRB10,genetic,False,NaN,1.0 µL,...,15-year-old stage,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_GRB10_CVCL_0042,trt_sh


In [44]:
obs['sig_id'] = obs.index

In [45]:
logger.info('  Enforcing strict obs schema')
obs_for_schema = enforce_obs_schema(obs)

2026-01-29 20:00:50 | [INFO]   Enforcing strict obs schema


In [47]:
logger.info('  Processing gene annotations')
var = process_gene_annotations(geneinfo, FULL_GENE_MATRIX)

2026-01-29 20:01:26 | [INFO]   Processing gene annotations
2026-01-29 20:01:26 | [INFO]   Restricting to 978 landmark genes
2026-01-29 20:01:31 | [INFO] Processed 50/978 entrez ids (50 mapped so far)
2026-01-29 20:01:36 | [INFO] Processed 100/978 entrez ids (100 mapped so far)
2026-01-29 20:01:42 | [INFO] Processed 150/978 entrez ids (150 mapped so far)
2026-01-29 20:01:47 | [INFO] Processed 200/978 entrez ids (200 mapped so far)
2026-01-29 20:01:52 | [INFO] Processed 250/978 entrez ids (250 mapped so far)
2026-01-29 20:01:57 | [INFO] Processed 300/978 entrez ids (300 mapped so far)
2026-01-29 20:02:02 | [INFO] Processed 350/978 entrez ids (350 mapped so far)
2026-01-29 20:02:07 | [INFO] Processed 400/978 entrez ids (400 mapped so far)
2026-01-29 20:02:12 | [INFO] Processed 450/978 entrez ids (450 mapped so far)
2026-01-29 20:02:17 | [INFO] Processed 500/978 entrez ids (500 mapped so far)
2026-01-29 20:02:22 | [INFO] Processed 550/978 entrez ids (550 mapped so far)
2026-01-29 20:02:27 

In [48]:
logger.info('  Matching obs to GCTX column IDs')
obs_helpers = build_obs_helpers(obs, PATHS["level5_gctx"])

2026-01-29 20:03:14 | [INFO]   Matching obs to GCTX column IDs
2026-01-29 20:03:16 | [INFO]   Loaded 473,647 column IDs from GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx
2026-01-29 20:03:17 | [INFO]   Using obs index to subset GCTX (473647/473647 samples match)


In [49]:
obs_helpers

,source_gctx,gctx_id
sig_id,,
AML001_CD34_24H:A05,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:A05
AML001_CD34_24H:A06,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:A06
AML001_CD34_24H:B05,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:B05
AML001_CD34_24H:B06,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:B06
AML001_CD34_24H:BRD-A03772856:0.37037,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:BRD-A03772856:0.37037
...,...,...
TAK004_U2OS_96H:TRCN0000370007:1,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,TAK004_U2OS_96H:TRCN0000370007:1
TAK004_U2OS_96H:TRCN0000370678:1,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,TAK004_U2OS_96H:TRCN0000370678:1
TAK004_U2OS_96H:TRCN0000370697:1,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,TAK004_U2OS_96H:TRCN0000370697:1


In [50]:
sig_raw[sig_raw['pert_type'].isin(['trt_cp']) | (sig_raw['pert_type'].isin(['ctl_vehicle']) & sig_raw['pert_iname'].isin(['DMSO']))]

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
0,AML001_CD34_24H:A05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:A05
1,AML001_CD34_24H:A06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:A06
2,AML001_CD34_24H:B05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...
3,AML001_CD34_24H:B06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:B06
4,AML001_CD34_24H:BRD-A03772856:0.37037,BRD-A03772856,BRD-A03772856,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...
...,...,...,...,...,...,...,...,...,...,...,...,...
469552,RAD001_PC3_6H:P02,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P02
469553,RAD001_PC3_6H:P13,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P13
469554,RAD001_PC3_6H:P23,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P23
469555,RAD001_PC3_6H:P24,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P24


In [51]:
logger.info('  Extracting expression from GCTX')
expr = load_expression(obs_helpers, PATHS["level5_gctx"], gene_ids=var.index.astype(str))

2026-01-29 20:03:54 | [INFO]   Extracting expression from GCTX


/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.10/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


In [52]:
logger.info('  Assembling AnnData')
padata_processed = assemble_anndata(obs_for_schema, var, expr)

2026-01-29 20:04:38 | [INFO]   Assembling AnnData


/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [53]:
logger.info(f'  Assembled AnnData: {padata_processed.n_obs:,} × {padata_processed.n_vars:,}')
logger.info('L1000-specific processing completed')

2026-01-29 20:05:36 | [INFO]   Assembled AnnData: 473,647 × 978
2026-01-29 20:05:36 | [INFO] L1000-specific processing completed


In [54]:
padata_processed.obs

,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,...,guide,dataset,assay,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts
sample_id,,,,,,,,,,,,,,,,,,,,,
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_BRD-A03772856_CD34,NaN,NaN,CD34,BRD-A03772856,compound,False,0.37037,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,3237298,-666,-666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
None_None_WWTR1_CVCL_0042,NaN,NaN,CVCL_0042,WWTR1,genetic,False,NaN,96.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,15-year-old stage,human,female,Caucasian,NaN,-666,-666
None_None_GRB10_CVCL_0042,NaN,NaN,CVCL_0042,GRB10,genetic,False,NaN,96.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,15-year-old stage,human,female,Caucasian,NaN,-666,-666
None_None_GRB10_CVCL_0042,NaN,NaN,CVCL_0042,GRB10,genetic,False,NaN,96.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,15-year-old stage,human,female,Caucasian,NaN,-666,-666


In [58]:
# Save the assembled dataset
logger.info(f"Saving assembled dataset to: {'./data/l1000_phase1/level5/' + config['output_file']}")
padata_processed.write_h5ad('./data/l1000_phase1/level5/' + config['output_file'], compression='gzip')
logger.info(f"{'./data/l1000_phase1/level5/' + config['output_file']} assembly complete")

2026-01-29 20:11:51 | [INFO] Saving assembled dataset to: ./data/l1000_phase1/level5/level5_phase1_not_filtered.h5ad
2026-01-29 20:13:39 | [INFO] ./data/l1000_phase1/level5/level5_phase1_not_filtered.h5ad assembly complete


In [59]:
padata_processed[padata_processed.obs['cell_type'] == 'CVCL_0566']

View of AnnData object with n_obs × n_vars = 273 × 978
    obs: 'plate', 'well', 'cell_type', 'perturbagen', 'pert_type', 'is_control', 'pert_dose_uM', 'pert_time_h', 'suspension_type', 'tissue', 'tissue_type', 'disease', 'library', 'stimulation', 'guide', 'dataset', 'assay', 'development_stage', 'organism', 'sex', 'self_reported_ethnicity', 'pubchem_cid', 'psbulk_cells', 'psbulk_counts'
    var: 'symbol'

In [60]:
padata_processed.obs[padata_processed.obs['is_control'] == True]

,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,...,guide,dataset,assay,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts
sample_id,,,,,,,,,,,,,,,,,,,,,
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,unknown,human,unknown,unknown,679,-666,-666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
None_None_lacZ_CVCL_0042,NaN,NaN,CVCL_0042,lacZ,genetic,True,NaN,96.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,15-year-old stage,human,female,Caucasian,NaN,-666,-666
None_None_LUCIFERASE_CVCL_0042,NaN,NaN,CVCL_0042,LUCIFERASE,genetic,True,NaN,96.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,15-year-old stage,human,female,Caucasian,NaN,-666,-666
None_None_LUCIFERASE_CVCL_0042,NaN,NaN,CVCL_0042,LUCIFERASE,genetic,True,NaN,96.0,cell,bone,...,NaN,LINCS_phase1_level3_epsilon,L1000 mRNA profiling assay,15-year-old stage,human,female,Caucasian,NaN,-666,-666


In [61]:
sig_raw[sig_raw['pert_iname'] == 'LUCIFERASE']

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
38964,CNS001_A375_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,A375,-666.0,-666,-666,96,h,96 h,DER001_A375_96H:TRCN0000072248:-666|KDA001_A37...
38975,CNS001_A549_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,A549,-666.0,-666,-666,96,h,96 h,KDB004_A549_96H:TRCN0000072254:-666|KDA001_A54...
38983,CNS001_ASC_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,ASC,-666.0,-666,-666,96,h,96 h,KDB001_ASC_96H:TRCN0000072250:-666|KDB008_ASC_...
39004,CNS001_HA1E_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,HA1E,-666.0,-666,-666,96,h,96 h,KDA010_HA1E_96H:TRCN0000072266:-666|KDA001_HA1...
39015,CNS001_HCC515_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,HCC515,-666.0,-666,-666,96,h,96 h,DER001_HCC515_96H:TRCN0000072248:-666|KDA005_H...
...,...,...,...,...,...,...,...,...,...,...,...,...
473006,TAK003_HEKTE_96H:TRCN0000072266:-666,TRCN0000072266,LUCIFERASE,ctl_vector,HEKTE,-666.0,-666,-666,96,h,96 h,TAK003_HEKTE_96H_X1_B7_DUO52HI53LO:H21|TAK003_...
473192,TAK003_PC3_96H:TRCN0000072266:-666,TRCN0000072266,LUCIFERASE,ctl_vector,PC3,-666.0,-666,-666,96,h,96 h,TAK003_PC3_96H_X1_B7_DUO52HI53LO:H21|TAK003_PC...
473450,TAK004_U2OS_96H:TRCN0000072246:1,TRCN0000072246,LUCIFERASE,ctl_vector,U2OS,1.0,µL,1 µL,96,h,96 h,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:M19|TAK004_U...
473451,TAK004_U2OS_96H:TRCN0000072248:1,TRCN0000072248,LUCIFERASE,ctl_vector,U2OS,1.0,µL,1 µL,96,h,96 h,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:M14|TAK004_U...
